# Генерация синтетических данных с использованием Faker

# Задание на самостоятельную работу
Создать синтетический набор данных, сохранить его в csv файлы, построить на основе данных онтологию, вывести онтограф.


### Вариант 4. Студенты и успеваемость
Сущности:
- Студент: id, ФИО, дата рождения, группа, год поступления, форма обучения.
- Группа: id, номер, курс, институт, направление подготовки.
- Предмет: id, название, семестр, количество часов, форма контроля (экзамен/зачёт), id преподавателя.
- Оценка: id, id студента, id предмета, дата, оценка (число или зачёт/незачёт).
- Преподаватель: id, ФИО, кафедра.

Объём: 10 групп по 25 студентов = 250 студентов, 30 предметов, у каждого студента оценки по всем предметам (≈ 7500 записей).

Онтология
- Классы: Студент, Группа, Предмет, Оценка, Преподаватель.
- Связи: студент учится в группе, группа изучает предметы (связка группа-предмет), преподаватель ведёт предмет, студент получает оценки.

In [64]:
from typing import Literal, Final
from dataclasses import dataclass
from datetime import date
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

N_GROUPS: Final[int] = 3
N_STUDENTS_IN_GROUP: Final[int] = 4
N_SUBJECTS: Final[int] = 5


@dataclass(frozen=True)
class Institute:
    institute: str
    departments: tuple[str, ...]

# С сайта МГТУ
INSTITUTES: Final[tuple[Institute, ...]] = (
    Institute(
        institute="Институт металлургии, машиностроения и материалообработки",
        departments=(
            "Кафедра литейных процессов и материаловедения",
            "Кафедра металлургии и химических технологий",
            "Кафедра обработки материалов давлением им. М.И. Бояршинова",
            "Кафедра машин и технологий обработки давлением и машиностроения",
            "Кафедра механики",
            "Кафедра проектирования и эксплуатации металлургических машин и оборудования",
        )
    ),
    Institute(
        institute="Институт горного дела и транспорта",
        departments=(
            "Кафедра геологии, маркшейдерского дела и обогащения полезных ископаемых",
            "Кафедра горных машин и транспортно-технологических комплексов",
            "Кафедра логистики и управления транспортными системами",
            "Кафедра разработки месторождений полезных ископаемых",
        )
    ),
    Institute(
        institute="Институт энергетики и автоматизированных систем",
        departments=(
            "Кафедра автоматизированного электропривода и мехатроники",
            "Кафедра теплотехнических и энергетических систем",
            "Кафедра электроники и микроэлектроники",
            "Кафедра электроснабжения промышленных предприятий",
            "Кафедра автоматизированных систем управления",
            "Кафедра бизнес-информатики и информационных технологий",
            "Кафедра вычислительной техники и программирования",
            "Кафедра информатики и информационной безопасности",
        )
    ),
    Institute(
        institute="Институт строительства, архитектуры и искусства",
        departments=(
            "Кафедра промышленного и гражданского строительства",
            "Кафедра урбанистики и инженерных систем",
            "Кафедра архитектуры и изобразительного искусства",
            "Кафедра дизайна",
            "Кафедра художественной обработки материалов",
        )
    ),
    Institute(
        institute="Институт экономики и управления",
        departments=(
            "Кафедра менеджмента и государственного управления",
            "Кафедра права и культурологии",
            "Кафедра философии",
            "Кафедра экономики",
        )
    ),
    Institute(
        institute="Институт гуманитарного образования",
        departments=(
            "Кафедра всеобщей истории",
            "Кафедра иностранных языков по техническим направлениям",
            "Кафедра лингвистики и перевода",
            "Кафедра русского языка как иностранного",
            "Кафедра русского языка, общего языкознания и массовой коммуникации",
            "Кафедра языкознания и литературоведения",
            "Кафедра дошкольного и специального образования",
            "Кафедра педагогического образования и документоведения",
            "Кафедра психологии",
            "Кафедра социальной работы и психолого-педагогического образования",
        )
    ),
    Institute(
        institute="Институт естествознания и стандартизации",
        departments=(
            "Кафедра промышленной экологии и безопасности жизнедеятельности",
            "Кафедра технологии, сертификации и сервиса автомобилей",
            "Кафедра химии",
            "Кафедра прикладной математики и информатики",
            "Кафедра физики",
        )
    ),
    Institute(
        institute="Факультет физической культуры и спортивного мастерства",
        departments=(
            "Кафедра спортивного совершенствования",
            "Кафедра физической культуры",
        )
    ),
)

# предметы по направлениям (примерные)
SUBJECT_POOL = {
    "Общие": [
        "История России", "Философия", "Иностранный язык",
        "Безопасность жизнедеятельности", "Физическая культура и спорт",
        "Русский язык и культура речи", "Правоведение"
    ],
    "Металлургия_Машиностроение": [
        "Теория металлургических процессов", "Металловедение",
        "Сопротивление материалов", "Начертательная геометрия",
        "Оборудование литейных цехов", "Проектирование цехов",
        "Механика жидкости и газа", "Термическая обработка металлов"
    ],
    "Горное_дело_Транспорт": [
        "Общая геология", "Маркшейдерия", "Проектирование карьеров",
        "Транспортная логистика", "Горные машины", "Геомеханика",
        "Обогащение полезных ископаемых", "Взрывное дело"
    ],
    "IT_Автоматизация": [
        "Объектно-ориентированное программирование", "Базы данных",
        "Операционные системы", "Сетевые технологии", "Архитектура ЭВМ",
        "Теория автоматического управления", "Информационная безопасность",
        "Алгоритмы и структуры данных"
    ],
    "Энергетика": [
        "Теоретические основы электротехники (ТОЭ)", "Электрические машины",
        "Электроснабжение предприятий", "Промышленная электроника",
        "Релейная защита", "Теплотехнические системы", "Электропривод"
    ],
    "Строительство_Архитектура": [
        "Строительная механика", "Архитектурное проектирование",
        "Железобетонные конструкции", "Урбанистика", "История архитектуры",
        "Дизайн-проектирование", "Инженерная геодезия", "Строительные материалы"
    ],
    "Экономика_Управление": [
        "Микроэкономика", "Макроэкономика", "Бухгалтерский учет и анализ",
        "Менеджмент", "Государственное управление", "Финансовый аудит",
        "Маркетинг", "Мировая экономика"
    ],
    "Гуманитарные_науки": [
        "Психология личности", "Методика преподавания", "Общее языкознание",
        "Теория перевода", "Социология", "Педагогика", "Литературоведение",
        "Документоведение"
    ],
    "Естествознание": [
        "Прикладная математика", "Общая физика", "Органическая химия",
        "Промышленная экология", "Метрология и стандартизация",
        "Математический анализ", "Экологический мониторинг"
    ],
    "Спорт": [
        "Теория и методика физкультуры", "Спортивная медицина",
        "Биомеханика", "Физиология человека", "Психология спорта",
        "Менеджмент в спорте"
    ]
}


@dataclass()
class Group:
    id: int
    number: int
    course: int
    institute: str  # Институт
    department: str  # Направление подготовки (по сути тоже кафедра)


@dataclass()
class Student:
    id: int
    first_name: str
    second_name: str
    last_name: str
    birth_date: date
    group: Group
    admission_year: int
    education_form: Literal["Очная", "Заочная"]


@dataclass()
class Subject:
    id: int
    name: str
    semester: int
    hours: int
    assessment_type: Literal["Экзамен", "Зачет"]  # Экзамен/Зачет
    lecturer_id: int  # id преподавателя


@dataclass()
class Grade:
    id: int
    student_id: int
    subject_id: int
    date: date
    grade: int | Literal["зачет", "Незачет"]


@dataclass()
class Lecturer:
    id: int
    first_name: str
    second_name: str
    last_name: str
    department: str  # кафедра


# Инициализация генератора
fake = Faker('ru_RU')
Faker.seed(42)  # фиксируем seed для воспроизводимости
random.seed(42)



## Генераторы данных

In [65]:

def get_realistic_subject(department_name: str, course: int) -> str:
    """
    Выбирает предмет в зависимости от кафедры и курса студента.
    1-2 курс: 70% шанс на общий предмет.
    3-4 курс: 90% шанс на профильный предмет.
    """
    dep = department_name.lower()

    # Определяем профиль
    if any(word in dep for word in ["металлург", "литей", "машин", "давлен", "механик"]):
        pool_key = "Металлургия_Машиностроение"
    elif any(word in dep for word in ["горн", "геолог", "транспорт", "логистик", "месторожд"]):
        pool_key = "Горное_дело_Транспорт"
    elif any(word in dep for word in ["автоматизир", "информат", "вычислит", "программ", "бизнес-информ"]):
        pool_key = "IT_Автоматизация"
    elif any(word in dep for word in ["энерг", "электро", "тепло"]):
        pool_key = "Энергетика"
    elif any(word in dep for word in ["строит", "архитект", "дизайн", "урбан"]):
        pool_key = "Строительство_Архитектура"
    elif any(word in dep for word in ["эконом", "менедж", "право", "управл"]):
        pool_key = "Экономика_Управление"
    elif any(word in dep for word in ["истори", "лингвист", "психолог", "педагог", "филолог", "язык"]):
        pool_key = "Гуманитарные_науки"
    elif any(word in dep for word in ["физик", "хими", "математ", "эколог", "сервис"]):
        pool_key = "Естествознание"
    elif any(word in dep for word in ["спорт", "физическ"]):
        pool_key = "Спорт"
    else:
        pool_key = "Общие"

    if course <= 2:
        # на младших курсах чаще общие предметы
        is_general = random.random() < 0.7
    else:
        # на старших курсах чаще профильные предметы
        is_general = random.random() < 0.1

    final_pool = SUBJECT_POOL["Общие"] if is_general else SUBJECT_POOL[pool_key]
    return random.choice(final_pool)


def generate_lecturers(n_per_department: int = 3) -> list[Lecturer]:
    lecturers = []
    l_id = 1
    for inst in INSTITUTES:
        for dep in inst.departments:
            for _ in range(n_per_department):
                gender = random.choice(['male', 'female'])
                if gender == 'male':
                    first_name, second_name, last_name = fake.first_name_male(), fake.middle_name_male(), fake.last_name_male()
                else:
                    first_name, second_name, last_name = fake.first_name_female(), fake.middle_name_female(), fake.last_name_female()

                lecturers.append(Lecturer(
                    id=l_id, first_name=first_name, second_name=second_name, last_name=last_name, department=dep
                ))
                l_id += 1
    return lecturers


def generate_subjects(groups: list[Group], lecturers: list[Lecturer], n_subjects_per_course: int = N_SUBJECTS) -> list[Subject]:
    subjects = []
    sub_id = 1

    # Собираем уникальные комбинации (кафедра, курс) из сгенерированных групп
    dep_course_pairs = sorted(list({(g.department, g.course) for g in groups}))

    for dep, course in dep_course_pairs:
        # Берем лекторов только текущей кафедры
        dep_lecturers = sorted([l for l in lecturers if l.department == dep], key=lambda x: x.id)

        # Генерируем ровно 30 предметов для данной связки
        for i in range(n_subjects_per_course):
            # Рассчитываем семестр на основе курса (1 курс -> 1,2 сем; 2 курс -> 3,4 сем и т.д.)
            semester = (course - 1) * 2 + (i % 2) + 1

            base_name = get_realistic_subject(department_name=dep, course=course)

            subject_name = f"{base_name} (Сем. {semester})"

            subjects.append(Subject(
                id=sub_id,
                name=subject_name,
                semester=semester,
                hours=random.choice([36 * h for h in range(1, 8)]),
                assessment_type=random.choice(["Экзамен", "Зачет"]),
                lecturer_id=random.choice(dep_lecturers).id
            ))
            sub_id += 1
    return subjects


def generate_groups(n_groups: int = N_GROUPS) -> list[Group]:
    groups = []
    for i in range(1, n_groups + 1):
        institute = random.choice(INSTITUTES)
        groups.append(Group(
            id=i,
            number=100 + i,
            course=random.randint(1, 4),
            institute=institute.institute,
            department=random.choice(institute.departments)
        ))
    return groups


def generate_students(groups: list[Group], students_per_group: int = N_STUDENTS_IN_GROUP) -> list[Student]:
    students = []
    s_id = 1
    for group in groups:
        for _ in range(students_per_group):
            gender = random.choice(['male', 'female'])
            if gender == 'male':
                first_name, second_name, last_name = fake.first_name_male(), fake.middle_name_male(), fake.last_name_male()
            else:
                first_name, second_name, last_name = fake.first_name_female(), fake.middle_name_female(), fake.last_name_female()

            students.append(Student(
                id=s_id,
                first_name=first_name,
                second_name=second_name,
                last_name=last_name,
                birth_date=fake.date_of_birth(minimum_age=17, maximum_age=25),
                group=group,
                admission_year=2026 - group.course,
                education_form=random.choice(["Очная", "Заочная"])
            ))
            s_id += 1
    return students


def generate_grades(students: list[Student], subjects: list[Subject], lecturers: list[Lecturer]) -> list[Grade]:
    grades = []
    grade_id = 1

    # маппинг id лектора и его кафедра
    lecturer_dep_map = {l.id: l.department for l in lecturers}

    # маппинг (кафедра, курс) в список из 30 предметов
    dep_course_subjects_map = {}
    for sub in subjects:
        dep = lecturer_dep_map[sub.lecturer_id]
        course = (sub.semester + 1) // 2
        key = (dep, course)
        if key not in dep_course_subjects_map:
            dep_course_subjects_map[key] = []
        dep_course_subjects_map[key].append(sub)


    # назначаем оценки студентам
    # сортируем студентов по ID перед генерацией оценок
    for student in sorted(students, key=lambda x: x.id):
        key = (student.group.department, student.group.course)

        student_subjects = sorted(dep_course_subjects_map.get(key, []), key=lambda x: x.id)

        for sub in student_subjects:
            if sub.assessment_type == "Экзамен":
                score = random.choice([2, 3, 4, 5])
            else:
                score = random.choice(["Зачет", "Незачет"])

            grades.append(Grade(
                id=grade_id,
                student_id=student.id,
                subject_id=sub.id,

                date=fake.date_between(start_date='-1y', end_date='today'),
                grade=score
            ))
            grade_id += 1

    return grades

## Генерируем всё

In [66]:

lecturers = generate_lecturers()

groups = generate_groups()

subjects = generate_subjects(groups, lecturers)

students = generate_students(groups)

grades = generate_grades(students, subjects, lecturers)

df_lecturers = pd.DataFrame(lecturers)
df_subjects = pd.DataFrame(subjects)
df_groups = pd.DataFrame(groups)
df_students = pd.DataFrame(students)
df_grades = pd.DataFrame(grades)
df_lecturers.to_csv("files/lecturers.csv", index=False)
df_subjects.to_csv("files/subjects.csv", index=False)
df_groups.to_csv("files/groups.csv", index=False)
df_students.to_csv("files/students.csv", index=False)
df_grades.to_csv("files/grades.csv", index=False)


# Вывод статистики
print(f"Сгенерировано преподавателей: {len(lecturers)}")
print(f"Сгенерировано предметов: {len(subjects)}")
print(f"Сгенерировано групп: {len(groups)}")
print(f"Сгенерировано студентов: {len(students)}")
print(f"Сгенерировано оценок: {len(grades)}")


Сгенерировано преподавателей: 132
Сгенерировано предметов: 15
Сгенерировано групп: 3
Сгенерировано студентов: 12
Сгенерировано оценок: 60


## Онтология


In [67]:
from owlready2 import *

# Создаем онтологию
onto = get_ontology("http://example.org/v1/university_ontology_V2.owl")

with onto:
    class Студент(Thing): pass
    class Группа(Thing): pass
    class Предмет(Thing): pass
    class Оценка(Thing): pass
    class Преподаватель(Thing): pass

    class учится_в(ObjectProperty):
        domain = [Студент]
        range = [Группа]

    class изучает(ObjectProperty):
        domain = [Группа]
        range = [Предмет]

    class ведет(ObjectProperty):
        domain = [Преподаватель]
        range = [Предмет]

    class получает_оценку(ObjectProperty):
        domain = [Студент]
        range = [Оценка]

    class оценка_по_предмету(ObjectProperty):
        domain = [Оценка]
        range = [Предмет]


# Словари для поиска созданных объектов
onto_groups = {}
onto_lecturers = {}
onto_subjects = {}
onto_students = {}

with onto:

    for g in groups:
        onto_g = Группа(f"Группа_{g.id}")
        # Заполняем атрибуты
        onto_g.номер_группы = [g.number]
        onto_g.курс = [g.course]
        onto_g.label = [f"Группа {g.number}"]
        onto_groups[g.id] = onto_g

    for l in lecturers:
        onto_l = Преподаватель(f"Преподаватель_{l.id}")
        fio_lecturer = f"{l.last_name} {l.first_name} {l.second_name}"
        onto_l.фио = [fio_lecturer]
        onto_l.label = [fio_lecturer]
        onto_lecturers[l.id] = onto_l

    # заполняем предметы и связываем их с преподавателями
    lecturer_dep_map = {l.id: l.department for l in lecturers}
    for sub in subjects:
        onto_sub = Предмет(f"Предмет_{sub.id}")
        onto_sub.название_предмета = [sub.name]
        onto_sub.семестр = [sub.semester]
        onto_sub.label = [sub.name]
        onto_subjects[sub.id] = onto_sub

        # Связь
        onto_lecturers[sub.lecturer_id].ведет.append(onto_sub)

    # Связываем группы с предметами
    for g in groups:
        for sub in subjects:
            sub_dep = lecturer_dep_map[sub.lecturer_id]
            sub_course = (sub.semester + 1) // 2
            if sub_dep == g.department and sub_course == g.course:
                onto_groups[g.id].изучает.append(onto_subjects[sub.id])

    # Заполняем студентов и связываем с группами
    for s in students:
        onto_s = Студент(f"Студент_{s.id}")
        fio_student = f"{s.last_name} {s.first_name} {s.second_name}"
        onto_s.фио = [fio_student]
        onto_s.label = [fio_student]
        onto_students[s.id] = onto_s

        # Связь
        onto_s.учится_в.append(onto_groups[s.group.id])

    # Заполняем оценки и связываем
    for gr in grades:
        onto_gr = Оценка(f"Оценка_{gr.id}")
        onto_gr.значение_оценки = [str(gr.grade)]

        sub_name = onto_subjects[gr.subject_id].название_предмета[0]
        onto_gr.label = [f"Оценка '{gr.grade}', {sub_name}"]

        # Связи
        onto_students[gr.student_id].получает_оценку.append(onto_gr)
        onto_gr.оценка_по_предмету.append(onto_subjects[gr.subject_id])


onto.save(file="university_ontology.owl", format="rdfxml")




In [68]:
test_student = onto_students[students[2].id]
print(f"\nПроверка")

print(f"Студент: {test_student.фио[0]}")
print(f"Учится в: {test_student.учится_в[0].label[0]}")

print("Первые 10 оценок:")
for grade in test_student.получает_оценку[:10]:
    предмет = grade.оценка_по_предмету[0].название_предмета[0]
    оценка = grade.значение_оценки[0]
    print(f" - {предмет}: {оценка}")


Проверка
Студент: Смирнов Кирилл Андреевич
Учится в: Группа 101
Первые 10 оценок:
 - Философия (Сем. 7): Зачет
 - Взрывное дело (Сем. 8): Незачет
 - Маркшейдерия (Сем. 7): Незачет
 - Взрывное дело (Сем. 8): 4
 - Маркшейдерия (Сем. 7): Зачет
 - Философия (Сем. 7): Зачет
 - Взрывное дело (Сем. 8): Незачет
 - Маркшейдерия (Сем. 7): Незачет
 - Взрывное дело (Сем. 8): 4
 - Маркшейдерия (Сем. 7): Зачет


# Онтограф через networkx

In [75]:
import networkx as nx
import matplotlib.pyplot as plt

Graph = nx.DiGraph()

# Обходим все объекты онтологии
for ind in onto.individuals():

    # Имя объекта для отображения
    label = ind.label[0] if ind.label else ind.name
    # Добавляем узел и записываем его класс, чтобы потом раскрасить
    Graph.add_node(ind.name, label=label, type=type(ind).__name__)

    # Ищем все объектные свойства
    for prop in ind.get_properties():
        if issubclass(prop, ObjectProperty):
            for target in prop[ind]:
                prop_name = prop.__name__
                # Добавляем стрелку от объекта к цели с именем свойства
                Graph.add_edge(ind.name, target.name, label=prop_name)

# цвета для разных классов
color_map = {
    "Студент": "#A1C9F4",
    "Группа": "#B3E2CD",
    "Предмет": "#F9C0C0",
    "Оценка": "#F9F1C0",
    "Преподаватель": "#D9F1FF"
}

# Получаем цвета для каждого узла графа
node_colors = [color_map.get(Graph.nodes[node].get("type"), "#D3D3D3") for node in Graph]
node_labels = nx.get_node_attributes(Graph, 'label')
edge_labels = nx.get_edge_attributes(Graph, 'label')

for node in Graph.nodes:
    node_type = Graph.nodes[node].get("type")
    Graph.nodes[node]['color'] = color_map.get(node_type, "#D3D3D3")

from pyvis.network import Network

net = Network(directed=True)

net.from_nx(Graph)

net.save_graph("university_graph.html")